## Imports

In [ ]:
import numpy as np
import xarray as xr
import copy
import src.utils
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cmocean
import os
import pathlib
import cartopy.crs as ccrs
import pandas as pd
import scipy.stats
import tqdm
import time
import xesmf as xe

## Set seaborn plotting style
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})
sns.set_palette("colorblind")

## initialize RNG
rng = np.random.default_rng(seed=100)

## Funcs

In [ ]:
## specify lons/lats for E/W boxes
LONS_W = [120, 200]
LONS_E = [200, 280]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-30, 0, 30],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[50, 120 -160],
            # xlocs=[],
            ylocs=[-30, 0, 30],
        )
        gl_.top_labels = False
        # gl_.bottom_labels = True
    # gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return

import cartopy.crs as ccrs


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


def plot_pr_diff(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance_r",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

def plot_sst_sigma(ax, data, lev_min=.3, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.amp",
        levels=np.arange(lev_min, lev_min+dx*(nlev+1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

def plot_pr(ax, data, lev_min=1, dx=1, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.rain",
        levels=np.arange(lev_min, lev_min+dx*(nlev+1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

## Data loading

In [ ]:
def sel_area(data, lon_range=None, lat_range=None):
    """select area on t grids"""

    ## find in lon range
    if lon_range is None:
        lon_idx = np.array([True for _ in data.nlon])
    else:
        lon_idx = (
            (data.TLONG >= lon_range[0]) & (data.TLONG <= lon_range[1])
        ).any("nlat").values

    if lat_range is None:
        lat_idx = np.array([True for _ in data.nlat])
    else:
        lat_idx = (
            (data.TLAT >= lat_range[0]) & (data.TLAT <= lat_range[1])
        ).any("nlon").values

    return data.isel(nlon=lon_idx, nlat=lat_idx)

def area_avg(data, varname, lon_range=None, lat_range=None):
    """average over area"""

    ## trim data
    data_ = sel_area(data, lon_range=lon_range, lat_range=lat_range)

    ## get dims to sum over
    integrate = lambda x : (x * data_["dA"]).sum(["nlon", "nlat"])

    return integrate(data_[varname]) / integrate(1.0)

def get_eli_helper(exceeds_thresh):
    """compute ELI from mask of threshold exceedance"""

    ## sum and count longitudes exceeding thresh
    lon = exceeds_thresh.longitude
    longitude_sum = (exceeds_thresh * lon).sum(["longitude", "latitude"])
    longitude_count = exceeds_thresh.sum(["longitude", "latitude"])

    ## eli is average longitude
    eli = longitude_sum / longitude_count

    return eli


def get_eli(rsst, rsst_thresh=0, max_lat=15):
    """compute ELI from tropical SST data"""

    ## get equatorial Pac. SST
    rsst_pac = rsst.sel(longitude=slice(120, 280), latitude=slice(-max_lat, max_lat))

    ## get boolean array where SST exceeds thresh
    exceeds_thresh0 = rsst_pac >= rsst_thresh

    ## compute initial ELI estimate
    eli0 = get_eli_helper(exceeds_thresh0)

    return eli0

## Load the data

### init. cluster

In [ ]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(n_workers=8)
client = Client(cluster)
client

### do loading

In [ ]:
SAVE_FP = pathlib.Path("/glade/work/kcarr/lme_data")
SAVE_FNAME = SAVE_FP / "pr.nc"

## load precip
pr = xr.open_dataarray(SAVE_FNAME).compute()
pr = pr.rename({"lon":"longitude","lat":"latitude"})

## convert m/s to mm/day
mm_per_m = 1e3
s_per_day = 24*60*60
pr *= mm_per_m * s_per_day

## Analysis

In [ ]:
## get climatology
get_clim = lambda x : x.groupby("time.month").mean(["time","member"])
clim = get_clim(pr.isel(time=slice(None,360)))

clim_fma = clim.sel(month=[2, 3, 4]).mean('month')
clim_aso = clim.sel(month=[8, 9, 10]).mean('month')

### Spatial plot

In [ ]:
## specify intervals
dxs = np.array([2])

fig = plt.figure(figsize=(7, 9), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=40, lon_range=(60, 280))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=1, format_func=format_func)


cp00 = plot_pr(axs[0, 0], clim_aso, dx=2)
cp00 = plot_pr(axs[1, 0], clim_fma, dx=2)
cp00 = plot_pr_diff(axs[2, 0], clim_aso-clim_fma, dx=2)

## label
axs[0, 0].set_title("(a) FMA", loc="left")
axs[1, 0].set_title("(b) ASO", loc="left")
axs[2, 0].set_title("(c) Difference", loc="left")

## add labels
# add_gridlines(axs)
bbox = dict(boxstyle="round", facecolor="white", alpha=1)
legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
for ax in axs.flatten():
    ax.set_aspect("auto")
    src.utils.plot_box(ax, lons=[65,150], lats=[-25,25], c="k", ls="--")

## colorbars
cb_kwargs = dict(label=r"$^{\circ}$C", pad=0.03)
# fig.colorbar(cp00, ax=axs[0,0], ticks=[-3.6,0,3.6], **cb_kwargs)
# fig.colorbar(cp10, ax=axs[1,0], ticks=[-1.8,0,1.8], **cb_kwargs)
add_gridlines(axs)
plt.show()

In [ ]:
## average over longitudes in australia/Asia
clim_meri = clim.sel(longitude=slice(65,140)).mean("longitude")

fig,ax = plt.subplots(figsize=(3,5))
ax.contourf(
    clim_meri.month,
    clim_meri.latitude,
    clim_meri.transpose("latitude","month"),
    cmap="cmo.rain",
    extend="both",
    levels=np.arange(0, 11, 1),
)

ax.set_ylim([-25,25])

src.utils.add_vticks([ax], xlines=[3, 9], xticks=[])
ax.set_xticks([3, 9], labels=["Mar", "Sep"])
ax.axhline(0, ls="--", c="gray", lw=.8)

plt.show()